## Remove old Weaviate DB files

In [ ]:
!rm -rf ~/.local/share/weaviate

# Approximate Nearest Neighbors

In [ ]:
from random import random, randint
from math import floor, log
import networkx as nx
import numpy as np
import matplotlib as mtplt
from matplotlib import pyplot as plt
from utils import *

vec_num = 40 # Number of vectors (nodes)
dim = 2 ## Dimention. Set to be 2. All the graph plots are for dim 2. If changed, then plots should be commented. 
m_nearest_neighbor = 2 # M Nearest Neigbor used in construction of the Navigable Small World (NSW)

vec_pos = np.random.uniform(size=(vec_num, dim))

### Query Vector

In [ ]:
## Query
query_vec = [0.5, 0.5]

nodes = []
nodes.append(("Q",{"pos": query_vec}))

G_query = nx.Graph()
G_query.add_nodes_from(nodes)

print("nodes = ", nodes, flush=True)

pos_query=nx.get_node_attributes(G_query,'pos')

## Brute Force

In [ ]:
(G_lin, G_best) = nearest_neigbor(vec_pos,query_vec)

pos_lin=nx.get_node_attributes(G_lin,'pos')
pos_best=nx.get_node_attributes(G_best,'pos')

fig, axs = plt.subplots()

nx.draw(G_lin, pos_lin, with_labels=True, node_size=150, node_color=[[0.8,0.8,1]], width=0.0, font_size=7, ax = axs)
nx.draw(G_query, pos_query, with_labels=True, node_size=200, node_color=[[0.5,0,0]], font_color='white', width=0.5, font_size=7, font_weight='bold', ax = axs)
nx.draw(G_best, pos_best, with_labels=True, node_size=200, node_color=[[0.85,0.7,0.2]], width=0.5, font_size=7, font_weight='bold', ax = axs)

### HNSW Construction

In [ ]:
GraphArray = construct_HNSW(vec_pos,m_nearest_neighbor)

for layer_i in range(len(GraphArray)-1,-1,-1):
    fig, axs = plt.subplots()

    print("layer_i = ", layer_i)
        
    if layer_i>0:
        pos_layer_0 = nx.get_node_attributes(GraphArray[0],'pos')
        nx.draw(GraphArray[0], pos_layer_0, with_labels=True, node_size=120, node_color=[[0.9,0.9,1]], width=0.0, font_size=6, font_color=(0.65,0.65,0.65), ax = axs)

    pos_layer_i = nx.get_node_attributes(GraphArray[layer_i],'pos')
    nx.draw(GraphArray[layer_i], pos_layer_i, with_labels=True, node_size=150, node_color=[[0.7,0.7,1]], width=0.5, font_size=7, ax = axs)
    nx.draw(G_query, pos_query, with_labels=True, node_size=200, node_color=[[0.8,0,0]], width=0.5, font_size=7, font_weight='bold', ax = axs)
    nx.draw(G_best, pos_best, with_labels=True, node_size=200, node_color=[[0.85,0.7,0.2]], width=0.5, font_size=7, font_weight='bold', ax = axs)
    plt.show()

### HNSW Search

In [ ]:
(SearchPathGraphArray, EntryGraphArray) = search_HNSW(GraphArray,G_query)

for layer_i in range(len(GraphArray)-1,-1,-1):
    fig, axs = plt.subplots()

    print("layer_i = ", layer_i)
    G_path_layer = SearchPathGraphArray[layer_i]
    pos_path = nx.get_node_attributes(G_path_layer,'pos')
    G_entry = EntryGraphArray[layer_i]
    pos_entry = nx.get_node_attributes(G_entry,'pos')

    if layer_i>0:
            pos_layer_0 = nx.get_node_attributes(GraphArray[0],'pos')
            nx.draw(GraphArray[0], pos_layer_0, with_labels=True, node_size=120, node_color=[[0.9,0.9,1]], width=0.0, font_size=6, font_color=(0.65,0.65,0.65), ax = axs)

    pos_layer_i = nx.get_node_attributes(GraphArray[layer_i],'pos')
    nx.draw(GraphArray[layer_i], pos_layer_i, with_labels=True, node_size=100, node_color=[[0.7,0.7,1]], width=0.5, font_size=6, ax = axs)
    nx.draw(G_path_layer, pos_path, with_labels=True, node_size=110, node_color=[[0.8,1,0.8]], width=0.5, font_size=6, ax = axs)
    nx.draw(G_query, pos_query, with_labels=True, node_size=80, node_color=[[0.8,0,0]], width=0.5, font_size=7, ax = axs)
    nx.draw(G_best, pos_best, with_labels=True, node_size=70, node_color=[[0.85,0.7,0.2]], width=0.5, font_size=7, ax = axs)
    nx.draw(G_entry, pos_entry, with_labels=True, node_size=80, node_color=[[0.1,0.9,0.1]], width=0.5, font_size=7, ax = axs)
    plt.show()

## Pure Vector Search with Weaviate (Python client v4)

This section uses the current Weaviate Python client API (`weaviate-client` v4). It stores the supplied vectors directly; Weaviate does **not** generate embeddings for this example.

Install/update the client if needed:

```python
%pip install -U weaviate-client
```


In [ ]:
import weaviate
import json
from weaviate.classes.config import Configure, DataType, Property, VectorDistances
from weaviate.classes.query import Filter, MetadataQuery


### Connect and create a collection

`connect_to_embedded()` starts a local embedded Weaviate instance. The first run may download the embedded server. If your environment does not support embedded Weaviate, connect to a running local server or Weaviate Cloud instead.

The collection uses cosine distance and self-provided vectors. Re-running this cell deletes the existing `MyCollection` collection so the example starts clean.


In [ ]:
client = weaviate.connect_to_embedded()

print("Weaviate ready:", client.is_ready())

collection_name = "MyCollection"

# CAUTION: delete this collection if it already exists.
if client.collections.exists(collection_name):
    client.collections.delete(collection_name)

collection = client.collections.create(
    name=collection_name,
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="foo", data_type=DataType.INT),
    ],
    vector_config=Configure.Vectors.self_provided(),
    vector_index_config=Configure.VectorIndex.hnsw(
        distance_metric=VectorDistances.COSINE
    ),
)

print("Created collection:", collection.name)


### Import the data

Each object has regular properties (`title`, `foo`) and a manually supplied six-dimensional vector. The vectors must have the same dimensionality as the query vector.


In [ ]:
data = [
    {
        "title": "First Object",
        "foo": 99,
        "vector": [0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
    },
    {
        "title": "Second Object",
        "foo": 77,
        "vector": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7],
    },
    {
        "title": "Third Object",
        "foo": 55,
        "vector": [0.3, 0.1, -0.1, -0.3, -0.5, -0.7],
    },
    {
        "title": "Fourth Object",
        "foo": 33,
        "vector": [0.4, 0.41, 0.42, 0.43, 0.44, 0.45],
    },
    {
        "title": "Fifth Object",
        "foo": 11,
        "vector": [0.5, 0.5, 0, 0, 0, 0],
    },
]

object_uuids = []
for item in data:
    uuid = collection.data.insert(
        properties={
            "title": item["title"],
            "foo": item["foo"],
        },
        vector=item["vector"],
    )
    object_uuids.append(uuid)

print(f"Inserted {len(object_uuids)} objects.")


In [ ]:
# Check the number of objects in the collection
aggregate = collection.aggregate.over_all(total_count=True)
print("Object count:", aggregate.total_count)


### Query Weaviate: vector search

`near_vector` finds the closest vectors to the supplied query vector. `limit=2` returns the two nearest objects.

The metadata request asks Weaviate to include each match's cosine distance and stored vector.


In [ ]:
query_vector = [-0.012, 0.021, -0.23, -0.42, 0.5, 0.5]

response = collection.query.near_vector(
    near_vector=query_vector,
    limit=2,
    return_metadata=MetadataQuery(distance=True, vector=True),
)

results = [
    {
        "title": obj.properties["title"],
        "distance": obj.metadata.distance,
        "uuid": str(obj.uuid),
        "vector": obj.vector,
    }
    for obj in response.objects
]

print(json.dumps(results, indent=2))


### Vector search with a property filter

This query searches by vector **and** filters the candidates to objects where `foo > 44`. The filter is applied by Weaviate as part of the search.


In [ ]:
filtered_response = collection.query.near_vector(
    near_vector=query_vector,
    filters=Filter.by_property("foo").greater_than(44),
    limit=2,
    return_metadata=MetadataQuery(distance=True),
    return_properties=["title", "foo"],
)

filtered_results = [
    {
        "title": obj.properties["title"],
        "foo": obj.properties["foo"],
        "distance": obj.metadata.distance,
        "uuid": str(obj.uuid),
    }
    for obj in filtered_response.objects
]

print(json.dumps(filtered_results, indent=2))


### `near_object` example

Instead of supplying a vector, `near_object` uses an existing object's UUID as the search reference. The example below uses the first object inserted above and returns up to three nearby objects.


In [ ]:
near_object_response = collection.query.near_object(
    near_object=object_uuids[0],
    limit=3,
    return_metadata=MetadataQuery(distance=True),
    return_properties=["title"],
)

near_object_results = [
    {
        "title": obj.properties["title"],
        "distance": obj.metadata.distance,
        "uuid": str(obj.uuid),
    }
    for obj in near_object_response.objects
]

print(json.dumps(near_object_results, indent=2))


### Close the client

Close the client when you have finished querying. If you want to run the queries again, rerun the connection and collection-creation cells above.


In [ ]:
client.close()
